In [32]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import math
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms

Dataset Loading

In [33]:
print("Loading CIFAR-10 via torchvision ...")
 
BATCH_SIZE = 64
 
# Standard CIFAR-10 channel-wise normalisation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std =(0.2470, 0.2435, 0.2616)),
])
 
train_dataset = CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_dataset  = CIFAR10(root="./data", train=False, download=True, transform=transform)
 
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"  Train samples: {len(train_dataset):,} | Test samples: {len(test_dataset):,}\n")

Loading CIFAR-10 via torchvision ...
  Train samples: 50,000 | Test samples: 10,000



#Prunable Layer

In [34]:
class PrunableLinear(nn.Module):
  
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
 
        # Standard weight & bias, initialised exactly as nn.Linear does
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
 
        # FIX 1: initialise gate_scores to -2.0 instead of 0.0
        # sigmoid(-2) ~ 0.12 -> gates start near-pruned, not half-open
        # This means the L1 penalty only needs a small push to reach 0,
        # while important gates are pulled open by the classification loss.
        self.gate_scores = nn.Parameter(
            torch.full((out_features, in_features), 0.5)
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # sigmoid maps gate_scores in R  ->  gates in (0, 1)
        gates = torch.sigmoid(self.gate_scores)   # shape: (out, in)
 
        # Element-wise mask: unimportant weights get multiplied near 0
        pruned_weights = self.weight * gates       # shape: (out, in)
 
        # From-scratch linear op — no F.linear or nn.Linear used.
        # Gradients flow through both self.weight and self.gate_scores via autograd.
        return x @ pruned_weights.T + self.bias
 
    def get_gates(self) -> torch.Tensor:
        """Return gate values (detached) for analysis."""
        return torch.sigmoid(self.gate_scores).detach()

#Network Definition

In [35]:
class SelfPruningNet(nn.Module):
   
    def __init__(self):
        super().__init__()
 
        # CNN feature extractor (standard Conv layers, no pruning here)
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
 
        # Prunable MLP classifier
        self.classifier = nn.Sequential(
            PrunableLinear(128 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            PrunableLinear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            PrunableLinear(256, 10),
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)
 
    def sparsity_loss(self) -> torch.Tensor:
        """L1 norm of all gate values — penalises open gates."""
        total = torch.tensor(0.0, device=next(self.parameters()).device)
        for module in self.modules():
            if isinstance(module, PrunableLinear):
                total = total + torch.sigmoid(module.gate_scores).abs().sum()
        return total
 
    def sparsity_stats(self, threshold: float = 1e-2) -> dict:
        """Percentage of weights whose gate < threshold (effectively pruned)."""
        pruned = total = 0
        for module in self.modules():
            if isinstance(module, PrunableLinear):
                gates = module.get_gates()
                pruned += int((gates < threshold).sum().item())
                total  += gates.numel()
        return {"pruned": pruned, "total": total,
                "sparsity_pct": 100.0 * pruned / total if total else 0.0}
 
    def all_gates(self) -> np.ndarray:
        """Flatten all gate values to a 1-D numpy array for plotting."""
        parts = []
        for module in self.modules():
            if isinstance(module, PrunableLinear):
                parts.append(module.get_gates().cpu().numpy().ravel())
        return np.concatenate(parts) if parts else np.array([])
 
    def gate_param_groups(self):
        """Return (gate_params, other_params) for separate lr scheduling."""
        gate_params  = [p for n, p in self.named_parameters() if "gate_scores" in n]
        other_params = [p for n, p in self.named_parameters() if "gate_scores" not in n]
        return gate_params, other_params

#Training & Evalustion helpers

In [36]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}\n")
 
 
def train_one_epoch(model, loader, optimizer, lam):
    model.train()
    total_loss = correct = seen = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits    = model(xb)
        cls_loss  = F.cross_entropy(logits, yb)
        spar_loss = model.sparsity_loss()
        # Total loss: classification + sparsity penalty
        loss = cls_loss + lam * spar_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        seen       += xb.size(0)
    return total_loss / seen, 100.0 * correct / seen

Running on: cpu



In [37]:
def evaluate(model, loader):
    model.eval()
    correct = seen = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            correct += (logits.argmax(1) == yb).sum().item()
            seen    += xb.size(0)
    return 100.0 * correct / seen

In [38]:
def run_experiment(lam: float, epochs: int = 8, lr: float = 3e-3):
    print(f"\n{'='*60}")
    print(f"  Lambda = {lam}  |  epochs = {epochs}  |  batch = {BATCH_SIZE}")
    print(f"{'='*60}")
 
    model = SelfPruningNet().to(DEVICE)
 
    # FIX 3: separate param groups — gate_scores get 3x higher lr
    # so they move fast enough to show sparsity within 8 epochs
    gate_params, other_params = model.gate_param_groups()
    optimizer = optim.Adam([
        {"params": other_params, "lr": lr,       "weight_decay": 1e-4},
        {"params": gate_params,  "lr": lr * 1.2, "weight_decay": 0.0},
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
 
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, lam)
        scheduler.step()
        # Print every epoch (only 8 total)
        te_acc = evaluate(model, test_loader)
        stats  = model.sparsity_stats()
        print(f"  Epoch {epoch:3d} | loss {tr_loss:.4f} | "
              f"train {tr_acc:.1f}% | test {te_acc:.1f}% | "
              f"sparse {stats['sparsity_pct']:.1f}%")
 
    final_acc  = evaluate(model, test_loader)
    final_stat = model.sparsity_stats()
    print(f"\n  Final test accuracy : {final_acc:.2f}%")
    print(f"  Sparsity level      : {final_stat['sparsity_pct']:.2f}%  "
          f"({final_stat['pruned']:,} / {final_stat['total']:,} weights pruned)")
 
    return model, final_acc, final_stat["sparsity_pct"]

#Lamda Experiments

In [39]:
LAMBDAS = [5e-5, 2e-4, 6e-4]   # low / medium / high
results  = {}
 
for lam in LAMBDAS:
    model, acc, sparsity = run_experiment(lam, epochs=8)
    results[lam] = (model, acc, sparsity)


  Lambda = 5e-05  |  epochs = 8  |  batch = 64
  Epoch   1 | loss 21.6198 | train 58.3% | test 69.6% | sparse 0.0%
  Epoch   2 | loss 6.8449 | train 70.1% | test 72.6% | sparse 0.0%
  Epoch   3 | loss 3.8030 | train 74.6% | test 75.3% | sparse 0.0%
  Epoch   4 | loss 2.7296 | train 78.1% | test 78.6% | sparse 0.0%
  Epoch   5 | loss 2.1868 | train 82.0% | test 79.5% | sparse 4.1%
  Epoch   6 | loss 1.8710 | train 85.2% | test 80.5% | sparse 25.2%
  Epoch   7 | loss 1.6722 | train 88.6% | test 82.2% | sparse 33.1%
  Epoch   8 | loss 1.5634 | train 91.0% | test 82.7% | sparse 35.2%

  Final test accuracy : 82.66%
  Sparsity level      : 35.16%  (415,619 / 1,182,208 weights pruned)

  Lambda = 0.0002  |  epochs = 8  |  batch = 64
  Epoch   1 | loss 78.6237 | train 57.0% | test 63.8% | sparse 0.0%
  Epoch   2 | loss 19.5577 | train 69.7% | test 68.8% | sparse 0.0%
  Epoch   3 | loss 8.8593 | train 74.6% | test 75.2% | sparse 0.0%
  Epoch   4 | loss 5.5413 | train 78.4% | test 76.9% | spar

#Result 

In [40]:
print("\n\n" + "="*60)
print("  RESULTS SUMMARY")
print("="*60)
print(f"  {'Lambda':<12} {'Test Accuracy':>15} {'Sparsity (%)':>14}")
print(f"  {'-'*44}")
for lam in LAMBDAS:
    _, acc, sparsity = results[lam]
    print(f"  {lam:<12} {acc:>14.2f}% {sparsity:>13.2f}%")
print("="*60)



  RESULTS SUMMARY
  Lambda         Test Accuracy   Sparsity (%)
  --------------------------------------------
  5e-05                 82.66%         35.16%
  0.0002                83.46%         72.16%
  0.0006                83.27%         92.22%


#Gate Distribution plot

In [41]:
best_lam = max(results, key=lambda l: results[l][1])
best_model, best_acc, best_sparsity = results[best_lam]
gate_vals = best_model.all_gates()
 
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor="#0d1117")
fig.suptitle("Self-Pruning Neural Network - Gate Value Analysis",
             color="white", fontsize=14, fontweight="bold", y=1.01)
 
# Left: histogram of gate values
ax = axes[0]
ax.set_facecolor("#161b22")
ax.hist(gate_vals, bins=100, color="#58a6ff", edgecolor="none", alpha=0.85)
ax.axvline(0.01, color="#f78166", linewidth=1.5, linestyle="--",
           label="Prune threshold (0.01)")
ax.set_xlabel("Gate Value", color="#c9d1d9")
ax.set_ylabel("Count", color="#c9d1d9")
ax.set_title(f"Gate Distribution  (lambda={best_lam})\n"
             f"Test acc: {best_acc:.1f}%  |  Sparsity: {best_sparsity:.1f}%",
             color="#c9d1d9")
ax.tick_params(colors="#c9d1d9")
ax.spines[:].set_color("#30363d")
ax.legend(facecolor="#161b22", labelcolor="#c9d1d9", edgecolor="#30363d")
 
# Right: accuracy vs sparsity bar chart across lambda values
ax2 = axes[1]
ax2.set_facecolor("#161b22")
lam_labels = [str(l) for l in LAMBDAS]
accs       = [results[l][1] for l in LAMBDAS]
sparsities = [results[l][2] for l in LAMBDAS]
 
x = np.arange(len(LAMBDAS))
w = 0.35
bars1 = ax2.bar(x - w/2, accs,       w, label="Test Accuracy (%)", color="#3fb950", alpha=0.85)
bars2 = ax2.bar(x + w/2, sparsities, w, label="Sparsity (%)",      color="#f78166", alpha=0.85)
 
ax2.set_xticks(x)
ax2.set_xticklabels(lam_labels, color="#c9d1d9")
ax2.set_xlabel("Lambda", color="#c9d1d9")
ax2.set_ylabel("Percentage (%)", color="#c9d1d9")
ax2.set_title("Accuracy vs Sparsity Trade-off", color="#c9d1d9")
ax2.tick_params(colors="#c9d1d9")
ax2.spines[:].set_color("#30363d")
ax2.legend(facecolor="#161b22", labelcolor="#c9d1d9", edgecolor="#30363d")
 
for bar in bars1:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{bar.get_height():.1f}", ha="center", va="bottom",
             fontsize=8, color="#c9d1d9")
for bar in bars2:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{bar.get_height():.1f}", ha="center", va="bottom",
             fontsize=8, color="#c9d1d9")
 
plt.tight_layout()
plt.savefig("gate_distribution.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.close()
print("\n  Plot saved -> gate_distribution.png")


  Plot saved -> gate_distribution.png
